## Install Dependencies


In [22]:
# Dependencis for the notebook
%pip install spacy pandas torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy
# optuna and optional dependencies
%pip install optuna scikit-learn plotly 
# Qwen optimization (optional) dependencies
%pip install flash-linear-attention causal-conv1d
# Mistral dependencies
%pip install accelerate


/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
import matplotlib.pyplot as plt
import matplotlib
import optuna.visualization
import optuna.importance
import json
from pathlib import Path
from collections import OrderedDict
import modules.llms as llm_parsers
import modules.deepparse_parser as deepparse
import modules.libpostal_client as libpostal_client
import modules.token_classifiers as token_classifiers
from modules.utils import compare_preds, format_time, natural_casing
from typing import NamedTuple
from tqdm.auto import tqdm

from typing import Callable

import abc
import contextlib
import pandas as pd
import time
import pprint
import textwrap
import numpy as np
import modules.train_ner as train_ner
import re

# Total number of addresses in the complete BZK data for calculating estimated time of processing
TOTAL_ADDRESSES = 4_394_539

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

ALL_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country"
]

BZK_ADDRESS_FIELDS = [
    'ApplicantCurrentAddress',
    'VictimBirthPlace',
    'VictimCurrentAddress',     
    'ApplicantBirthPlace',
    'VictimDeathPlace'
]

## Set hardware accelerator


In [24]:
# List available hardware accelerators for PyTorch
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe


In [25]:
if available_accelerators:
    device = available_accelerators[0]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:0


## Load Dataset

Read all splits and concat them

In [26]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_merged : pd.DataFrame = pd.concat(
    [bzkopen_train, bzkopen_val, bzkopen_test], 
    keys=['train', 'val', 'test'], 
    names=['split', 'row']
)
display(bzkopen_merged.sample(5))

field                        FullAddress  \
split row                                                               
train 170      ApplicantBirthPlace                                Lom   
val   48          VictimBirthPlace                          Hohenrode   
train 364      ApplicantBirthPlace                    Frankfurt/Main.   
      120  ApplicantCurrentAddress  Fretter Nr. 70, Reg.Bez. Arnsberg   
      341         VictimBirthPlace                     Warschau/Polen   

          UnitNumber HouseNumber StreetName Neighborhood            City  \
split row                                                                  
train 170        NaN         NaN        NaN          NaN             Lom   
val   48         NaN         NaN        NaN          NaN       Hohenrode   
train 364        NaN         NaN        NaN          NaN  Frankfurt/Main   
      120        NaN          70        NaN          NaN         Fretter   
      341        NaN         NaN        NaN          NaN        Warschau   

           District Region State Country PostalCode  
split row                                            
train 170       NaN    NaN   NaN     NaN        NaN  
val   48        NaN    NaN   NaN     NaN        NaN  
train 364       NaN   Main   NaN     NaN        NaN  
      120  Arnsberg    NaN   NaN     NaN        NaN  
      341       NaN    NaN   NaN   Polen        NaN

## Create folds



In [27]:
N_FOLDS = 10

# Set shuffle seed for reproducibility
SEED=4841762


In [28]:
# Define paths for saving models and predictions
models_path = Path("models") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
models_path.mkdir(parents=True, exist_ok=True)
preds_path = Path("experiments_data") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
preds_path.mkdir(parents=True, exist_ok=True)

In [29]:
class Fold(NamedTuple):
    fold_idx: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame
    models_path: Path
    preds_path: Path

In [30]:

shuffled_bzkopen = bzkopen_merged.sample(frac=1, random_state=SEED)

fold_indices = np.array_split(shuffled_bzkopen.index, N_FOLDS)

folds = []

for fold_idx, val_indices in enumerate(fold_indices):
    train_indices = shuffled_bzkopen.index.difference(val_indices)
    fold_models_path = models_path / f"fold_{fold_idx}"
    fold_models_path.mkdir(parents=True, exist_ok=True)
    fold_preds_path = preds_path / f"fold_{fold_idx}"
    fold_preds_path.mkdir(parents=True, exist_ok=True)
    folds.append(Fold(
        fold_idx, 
        shuffled_bzkopen.loc[train_indices], 
        shuffled_bzkopen.loc[val_indices],
        fold_models_path, 
        fold_preds_path
    ))
display(pd.DataFrame(
    [[len(x) for x in [fold.val_data, fold.train_data]] for fold in folds], 
    columns=['val_fold_size', 'train_fold_size'])
)

,val_fold_size,train_fold_size
0,108,967
1,108,967
2,108,967
3,108,967
4,108,967
5,107,968
6,107,968
7,107,968
8,107,968
9,107,968


# Train Spacy NER models

These models are used for the pattern example matching strategy so they will be reused over multiple different experiments. Therefore it is useful to train them in advance.

In [31]:
for fold in tqdm(folds, unit="fold"):
    output_dir = fold.models_path / 'ner_bzk'
    if output_dir.exists():
        print(f"Fold {fold.fold_idx} already has a trained model, skipping training.")
        continue
    train_ner.train(
        n_iter=30,
        output_dir=output_dir,
        train_df=fold.train_data,
        val_df=fold.val_data
    )

  0%|          | 0/10 [00:00<?, ?fold/s]

Fold 0 already has a trained model, skipping training.
Fold 1 already has a trained model, skipping training.
Fold 2 already has a trained model, skipping training.
Fold 3 already has a trained model, skipping training.
Fold 4 already has a trained model, skipping training.
Fold 5 already has a trained model, skipping training.
Fold 6 already has a trained model, skipping training.
Fold 7 already has a trained model, skipping training.
Fold 8 already has a trained model, skipping training.
Fold 9 already has a trained model, skipping training.


In [32]:
# Metrics on the required entities (Country, City, StreetName, HouseNumber)
required_results = OrderedDict()
specific_results = OrderedDict()


class FoldEvaluator:
    def __init__(self, model_name : str, supported_entities : list[str]):
        # Metrics on the required entities (Country, City, StreetName, HouseNumber)
        self._required_metric_records = None
        self._specific_metric_records = None
        self.required_metrics = None
        self.specific_metrics = None
        self.model_name = model_name
        self.supported_entities = supported_entities
    
    def __enter__(self):
        self.required_metrics = None
        self.specific_metrics = None
        self._required_metric_records = []
        self._specific_metric_records = []
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        global required_results, specific_results
        self.required_metrics = pd.DataFrame(self._required_metric_records).aggregate(['mean', 'std']).unstack()
        specific_metrics_df = pd.DataFrame(self._specific_metric_records)
        self.specific_metrics = pd.DataFrame(self._specific_metric_records).aggregate(['mean', 'std']).unstack()
        required_results[self.model_name] = self.required_metrics
        specific_results[self.model_name] = self.specific_metrics
        print(f"{self.model_name} - Metrics for Country City Street House:")
        display(self.required_metrics)
        print(f"{self.model_name} - Specific Metrics:")
        display(self.specific_metrics.unstack(level=0))
        self._required_metric_records = None
        self._specific_metric_records = None

    def _record_predictions(self, fold : Fold, preds : list[dict]):
        preds_df = pd.DataFrame(preds)
        fold_metrics = compare_preds(preds_df, fold.val_data, target_columns=REQUIRED_ENTITIES)
        self._required_metric_records.append(pd.Series(fold_metrics))
        specific_fold_metrics = OrderedDict()
        for entity in self.supported_entities:
            entity_metrics = compare_preds(preds_df, fold.val_data, target_columns=[entity])
            specific_fold_metrics[entity] = pd.Series(entity_metrics)
        for bzk_field in BZK_ADDRESS_FIELDS:
            mask = fold.val_data['field'] == bzk_field
            field_metrics = compare_preds(preds_df[mask.reset_index(drop=True)], fold.val_data[mask], target_columns=REQUIRED_ENTITIES)
            specific_fold_metrics[bzk_field] = pd.Series(field_metrics)
        self._specific_metric_records.append(pd.concat(specific_fold_metrics, names=['Field/Entity', 'Metric']))

    def load_preds_and_skip(self, folds : list[Fold]) -> list[Fold]:
        """
        For each fold, load its predictions if available on disk.
        Returns the folds for which predictions were not loaded
        """
        unloaded_folds = []
        for fold in folds:
            preds_file = fold.preds_path / self.model_name / "preds.json"
            if preds_file.exists():
                print(f"Fold {fold.fold_idx} already has predictions for {self.model_name}, loading from file.")
                with open(preds_file, "r") as f:
                    preds = json.load(f)
                self._record_predictions(fold, preds)
            else:
                unloaded_folds.append(fold)
        return unloaded_folds
            
    
    def run_on_fold(self, fold : Fold, model):
        assert self._required_metric_records is not None and self._specific_metric_records is not None, "FoldMetricRecorder must be used as a context manager"
        preds_file = fold.preds_path / self.model_name / "preds.json"
        preds_file.parent.mkdir(parents=True, exist_ok=True)
        preds = model.parse_addresses(fold.val_data['FullAddress'].tolist())
        with open(preds_file, "w") as f:
            json.dump(preds, f, indent=4)
        self._record_predictions(fold, preds)
    

In [33]:
with FoldEvaluator("libpostal", libpostal_client.LIBPOSTAL_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = libpostal_client.LibpostalClient()
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        model.close()

Fold 0 already has predictions for libpostal, loading from file.
Fold 1 already has predictions for libpostal, loading from file.
Fold 2 already has predictions for libpostal, loading from file.
Fold 3 already has predictions for libpostal, loading from file.
Fold 4 already has predictions for libpostal, loading from file.
Fold 5 already has predictions for libpostal, loading from file.
Fold 6 already has predictions for libpostal, loading from file.
Fold 7 already has predictions for libpostal, loading from file.
Fold 8 already has predictions for libpostal, loading from file.
Fold 9 already has predictions for libpostal, loading from file.
libpostal - Metrics for Country City Street House:


accuracy                   mean    0.687913
                           std     0.032383
precision                  mean    0.637836
                           std     0.045664
recall                     mean    0.419072
                           std     0.041448
f1                         mean    0.505555
                           std     0.043244
accuracy_with_tol_1        mean    0.698390
                           std     0.033153
accuracy_with_tol_2        mean    0.706533
                           std     0.030905
accuracy_with_tol_3        mean    0.721655
                           std     0.029771
accuracy_with_tol_4        mean    0.742580
                           std     0.032508
average_levenshtein        mean    2.488696
                           std     0.277225
average_similarity         mean    0.728394
                           std     0.028393
average_levenshtein_match  mean    3.200120
                           std     0.437061
average_similarity_match   mean 

libpostal - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City     State  \
Metric                                                                         
accuracy                  mean     0.869756    0.771218   0.319064  0.948884   
                          std      0.038574    0.045804   0.043952  0.022337   
precision                 mean     0.748952    0.533138   0.615605  0.667500   
                          std      0.045200    0.055681   0.096090  0.244143   
recall                    mean     0.679602    0.524948   0.299415  0.468452   
                          std      0.063833    0.090766   0.047211  0.175110   
f1                        mean     0.712181    0.528305   0.401641  0.536922   
                          std      0.053743    0.072998   0.058985  0.181123   
accuracy_with_tol_1       mean     0.881862    0.774013   0.336743  0.951670   
                          std      0.039570    0.047422   0.043739  0.020808   
accuracy_with_tol_2       mean     0.906057    0.774013   0.337677  0.952596   
                          std      0.025728    0.047422   0.042482  0.021559   
accuracy_with_tol_3       mean     0.923728    0.780512   0.348849  0.955400   
                          std      0.023133    0.047762   0.039812  0.023401   
accuracy_with_tol_4       mean     0.950727    0.793510   0.374879  0.976791   
                          std      0.023992    0.048253   0.043310  0.017032   
average_levenshtein       mean     0.582321    2.770093   5.511111  0.272300   
                          std      0.219950    0.545948   0.370673  0.116759   
average_similarity        mean     0.894377    0.836213   0.382294  0.951692   
                          std      0.032673    0.033275   0.038977  0.021046   
average_levenshtein_match mean     0.616913    3.066767  12.104832  0.288211   
                          std      0.244294    0.665067   1.730728  0.130935   
average_similarity_match  mean     0.940524    0.921863   0.831204  0.999054   
                          std      0.016100    0.020669   0.047889  0.002024   
no_match_rate             mean     0.049308    0.093025   0.539607  0.047404   
                          std      0.020619    0.025561   0.043935  0.021112   

Field/Entity                     Country  PostalCode  ApplicantCurrentAddress  \
Metric                                                                          
accuracy                  mean  0.791615    0.468354                 0.529093   
                          std   0.040104    0.493989                 0.054128   
precision                 mean  0.740418    0.033333                 0.639412   
                          std   0.122392    0.105409                 0.071530   
recall                    mean  0.317047    0.016667                 0.411403   
                          std   0.092088    0.052705                 0.070355   
f1                        mean  0.441058    0.022222                 0.500029   
                          std   0.109171    0.070273                 0.072820   
accuracy_with_tol_1       mean  0.800943    0.468354                 0.540611   
                          std   0.040885    0.493989                 0.053290   
accuracy_with_tol_2       mean  0.808385    0.473001                 0.553091   
                          std   0.043867    0.498847                 0.047712   
accuracy_with_tol_3       mean  0.833532    0.473001                 0.578030   
                          std   0.044589    0.498847                 0.043286   
accuracy_with_tol_4       mean  0.851203    0.475805                 0.615146   
                          std   0.045661    0.501819                 0.045510   
average_levenshtein       mean  1.091260    0.501237                 3.955788   
                          std   0.255998    0.262921                 0.495107   
average_similarity        mean  0.800691    0.468974                 0.592596   
                          std   0.041506    0.494633      

In [34]:
with FoldEvaluator("deepparse", deepparse.DEEPPARSE_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = deepparse.DeepParseParser(device=device)
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for deepparse, loading from file.
Fold 1 already has predictions for deepparse, loading from file.
Fold 2 already has predictions for deepparse, loading from file.
Fold 3 already has predictions for deepparse, loading from file.
Fold 4 already has predictions for deepparse, loading from file.
Fold 5 already has predictions for deepparse, loading from file.
Fold 6 already has predictions for deepparse, loading from file.
Fold 7 already has predictions for deepparse, loading from file.
Fold 8 already has predictions for deepparse, loading from file.
Fold 9 already has predictions for deepparse, loading from file.
deepparse - Metrics for Country City Street House:


accuracy                   mean    0.570431
                           std     0.031744
precision                  mean    0.321375
                           std     0.033958
recall                     mean    0.271941
                           std     0.035722
f1                         mean    0.294381
                           std     0.034086
accuracy_with_tol_1        mean    0.577862
                           std     0.031669
accuracy_with_tol_2        mean    0.593216
                           std     0.026731
accuracy_with_tol_3        mean    0.609724
                           std     0.024234
accuracy_with_tol_4        mean    0.630428
                           std     0.027539
average_levenshtein        mean    3.934809
                           std     0.331924
average_similarity         mean    0.648879
                           std     0.026140
average_levenshtein_match  mean    5.110830
                           std     0.502228
average_similarity_match   mean 

deepparse - Specific Metrics:


Field/Entity                    HouseNumber       City   Country  PostalCode  \
Metric                                                                         
accuracy                  mean     0.864183   0.253037  0.692073    0.922793   
                          std      0.037154   0.040026  0.041137    0.022864   
precision                 mean     0.834101   0.318456  0.272803    0.025000   
                          std      0.053190   0.056522  0.155691    0.079057   
recall                    mean     0.656874   0.255519  0.079102    0.016667   
                          std      0.079146   0.040036  0.041755    0.052705   
f1                        mean     0.733792   0.283273  0.118521    0.020000   
                          std      0.065601   0.046228  0.056528    0.063246   
accuracy_with_tol_1       mean     0.886483   0.259536  0.692999    0.928375   
                          std      0.039400   0.035509  0.039892    0.022018   
accuracy_with_tol_2       mean     0.939529   0.264183  0.694860    0.939521   
                          std      0.020732   0.035147  0.038122    0.020803   
accuracy_with_tol_3       mean     0.963699   0.271625  0.725554    0.948840   
                          std      0.016121   0.035842  0.039199    0.019791   
accuracy_with_tol_4       mean     0.976739   0.292108  0.763690    0.956308   
                          std      0.010060   0.038645  0.043218    0.020090   
average_levenshtein       mean     0.453141   6.726912  1.825675    0.540247   
                          std      0.168387   0.503971  0.308215    0.204010   
average_similarity        mean     0.874064   0.405451  0.697264    0.923041   
                          std      0.034185   0.030721  0.041245    0.022868   
average_levenshtein_match mean     0.512898  10.050439  2.631671    0.588115   
                          std      0.199188   0.779379  0.570586    0.233693   
average_similarity_match  mean     0.980704   0.605467  0.994171    0.997275   
                          std      0.010189   0.043294  0.003977    0.004388   
no_match_rate             mean     0.108818   0.330270  0.298615    0.074420   
                          std      0.031054   0.021273  0.041969    0.023277   

Field/Entity                    ApplicantCurrentAddress  VictimBirthPlace  \
Metric                                                                      
accuracy                  mean                 0.352289          0.763075   
                          std                  0.026584          0.052177   
precision                 mean                 0.309011          0.414742   
                          std                  0.054827          0.087619   
recall                    mean                 0.230186          0.412114   
                          std                  0.045794          0.096504   
f1                        mean                 0.263626          0.412542   
                          std                  0.049411          0.089687   
accuracy_with_tol_1       mean                 0.365328          0.765249   
                          std                  0.022965          0.054969   
accuracy_with_tol_2       mean                 0.389361          0.765249   
                          std                  0.022312          0.054969   
accuracy_with_tol_3       mean                 0.419085          0.775304   
                          std                  0.014867          0.045790   
accuracy_with_tol_4       mean                 0.449611          0.787457   
                          std                  0.015847          0.042256   
average_levenshtein       mean                 6.415092          1.823489   
                          std                  0.416029          0.424430   
average_similarity        mean                 0.466432          0.810860   
                          std                  0.027902          0.044962   
average_levenshtein_match mean                 9.573635          

In [35]:
with FoldEvaluator("xlm-roberta-large", token_classifiers.FIGHTING_CRIME_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = token_classifiers.TokenClassifierAddressParser(
            model_name="hm-haitham/xlm-roberta-large-address-parser", 
            device=device,
            aggregation_strategy="average"
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for xlm-roberta-large, loading from file.
Fold 1 already has predictions for xlm-roberta-large, loading from file.


Fold 2 already has predictions for xlm-roberta-large, loading from file.
Fold 3 already has predictions for xlm-roberta-large, loading from file.
Fold 4 already has predictions for xlm-roberta-large, loading from file.
Fold 5 already has predictions for xlm-roberta-large, loading from file.
Fold 6 already has predictions for xlm-roberta-large, loading from file.
Fold 7 already has predictions for xlm-roberta-large, loading from file.
Fold 8 already has predictions for xlm-roberta-large, loading from file.
Fold 9 already has predictions for xlm-roberta-large, loading from file.
xlm-roberta-large - Metrics for Country City Street House:


accuracy                   mean    0.805393
                           std     0.017126
precision                  mean    0.709732
                           std     0.037648
recall                     mean    0.657383
                           std     0.032906
f1                         mean    0.682448
                           std     0.033984
accuracy_with_tol_1        mean    0.816312
                           std     0.014871
accuracy_with_tol_2        mean    0.822359
                           std     0.013141
accuracy_with_tol_3        mean    0.833511
                           std     0.014613
accuracy_with_tol_4        mean    0.848875
                           std     0.016481
average_levenshtein        mean    1.430642
                           std     0.191547
average_similarity         mean    0.840967
                           std     0.012325
average_levenshtein_match  mean    1.589168
                           std     0.215828
average_similarity_match   mean 

xlm-roberta-large - Specific Metrics:


Field/Entity                     Country     State      City  StreetName  \
Metric                                                                     
accuracy                  mean  0.909770  0.925632  0.517272    0.874507   
                          std   0.023154  0.029648  0.034361    0.035855   
precision                 mean  0.799440  0.287460  0.599839    0.746411   
                          std   0.098036  0.214509  0.049758    0.078634   
recall                    mean  0.795060  0.227500  0.520498    0.741961   
                          std   0.074286  0.148682  0.038023    0.088406   
f1                        mean  0.795242  0.250404  0.557059    0.743500   
                          std   0.074995  0.170906  0.041079    0.080808   
accuracy_with_tol_1       mean  0.922785  0.932139  0.519133    0.886587   
                          std   0.020567  0.031562  0.036684    0.030620   
accuracy_with_tol_2       mean  0.927440  0.934000  0.520059    0.886587   
                          std   0.020439  0.032290  0.036783    0.030620   
accuracy_with_tol_3       mean  0.937660  0.947015  0.533091    0.894946   
                          std   0.019177  0.028762  0.035266    0.034307   
accuracy_with_tol_4       mean  0.958143  0.964694  0.550762    0.900537   
                          std   0.014660  0.021348  0.034051    0.033458   
average_levenshtein       mean  0.419514  0.410012  3.989339    1.055114   
                          std   0.114472  0.217234  0.416626    0.360446   
average_similarity        mean  0.916798  0.931931  0.602209    0.906168   
                          std   0.021278  0.031553  0.029204    0.024307   
average_levenshtein_match mean  0.458388  0.441647  5.128576    1.129630   
                          std   0.133956  0.240395  0.613353    0.396491   
average_similarity_match  mean  0.996535  0.989823  0.772910    0.967412   
                          std   0.004156  0.004951  0.032848    0.021041   
no_match_rate             mean  0.080002  0.058550  0.220535    0.063240   
                          std   0.021531  0.029339  0.029331    0.018911   

Field/Entity                    HouseNumber  ApplicantCurrentAddress  \
Metric                                                                 
accuracy                  mean     0.920024                 0.706104   
                          std      0.018625                 0.036906   
precision                 mean     0.844150                 0.717041   
                          std      0.046113                 0.058461   
recall                    mean     0.799598                 0.648604   
                          std      0.040271                 0.050138   
f1                        mean     0.821117                 0.680719   
                          std      0.041545                 0.050893   
accuracy_with_tol_1       mean     0.936743                 0.725455   
                          std      0.012259                 0.034663   
accuracy_with_tol_2       mean     0.955348                 0.732211   
                          std      0.011438                 0.035077   
accuracy_with_tol_3       mean     0.968345                 0.752802   
                          std      0.011017                 0.033414   
accuracy_with_tol_4       mean     0.986059                 0.777638   
                          std      0.007881                 0.036337   
average_levenshtein       mean     0.258602                 2.176361   
                          std      0.059059                 0.337750   
average_similarity        mean     0.938691                 0.758700   
                          std      0.011627                 0.027834   
average_levenshtein_match mean     0.268146                 2.522452   
                          std      0.062575                 0.402124   
average_similarity_match  mean     0.972128                 0.878942   
                          std      0.007887                 0.033107   

In [36]:
with FoldEvaluator("SpaCy", supported_entities=ALL_ENTITIES) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        for fold in tqdm(folds_to_process, unit="fold"):
            model = train_ner.SpaCyAddressParser(model_dir=fold.models_path / 'ner_bzk')
            evaluator.run_on_fold(fold, model)

Fold 0 already has predictions for SpaCy, loading from file.
Fold 1 already has predictions for SpaCy, loading from file.
Fold 2 already has predictions for SpaCy, loading from file.
Fold 3 already has predictions for SpaCy, loading from file.
Fold 4 already has predictions for SpaCy, loading from file.
Fold 5 already has predictions for SpaCy, loading from file.
Fold 6 already has predictions for SpaCy, loading from file.
Fold 7 already has predictions for SpaCy, loading from file.
Fold 8 already has predictions for SpaCy, loading from file.
Fold 9 already has predictions for SpaCy, loading from file.
SpaCy - Metrics for Country City Street House:


accuracy                   mean    0.889339
                           std     0.029299
precision                  mean    0.853173
                           std     0.032473
recall                     mean    0.814881
                           std     0.044588
f1                         mean    0.833505
                           std     0.038487
accuracy_with_tol_1        mean    0.898163
                           std     0.030274
accuracy_with_tol_2        mean    0.906304
                           std     0.027338
accuracy_with_tol_3        mean    0.911193
                           std     0.025994
accuracy_with_tol_4        mean    0.917709
                           std     0.025050
average_levenshtein        mean    0.891310
                           std     0.339198
average_similarity         mean    0.912928
                           std     0.024751
average_levenshtein_match  mean    0.953698
                           std     0.385043
average_similarity_match   mean 

SpaCy - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.902328    0.878202      0.893129   
                          std      0.032101    0.043076      0.038305   
precision                 mean     0.816794    0.807996      0.511732   
                          std      0.075382    0.053107      0.297710   
recall                    mean     0.778129    0.729742      0.332692   
                          std      0.062919    0.082073      0.157522   
f1                        mean     0.796591    0.766370      0.379130   
                          std      0.066641    0.068791      0.179513   
accuracy_with_tol_1       mean     0.924628    0.880054      0.893129   
                          std      0.033647    0.043758      0.038305   
accuracy_with_tol_2       mean     0.950692    0.881906      0.893129   
                          std      0.018680    0.043695      0.038305   
accuracy_with_tol_3       mean     0.958134    0.885618      0.894055   
                          std      0.018275    0.044373      0.037803   
accuracy_with_tol_4       mean     0.964642    0.894929      0.895916   
                          std      0.015765    0.045907      0.036132   
average_levenshtein       mean     0.456983    1.250511      1.102302   
                          std      0.222889    0.636207      0.394527   
average_similarity        mean     0.925193    0.905153      0.895510   
                          std      0.024794    0.036524      0.037790   
average_levenshtein_match mean     0.479478    1.359887      1.241778   
                          std      0.237589    0.745530      0.468732   
average_similarity_match  mean     0.965501    0.970893      0.996440   
                          std      0.014344    0.016226      0.005109   
no_match_rate             mean     0.041866    0.067870      0.101289   
                          std      0.014062    0.029257      0.037580   

Field/Entity                        City  District     State    Region  \
Metric                                                                   
accuracy                  mean  0.815040  0.931187  0.950684  0.000000   
                          std   0.055656  0.030649  0.015281  0.000000   
precision                 mean  0.859895  0.695310  0.736667  0.000000   
                          std   0.049514  0.155584  0.233967  0.000000   
recall                    mean  0.839322  0.719165  0.413333  0.000000   
                          std   0.044755  0.164600  0.271660  0.000000   
f1                        mean  0.849330  0.696681  0.484380  0.000000   
                          std   0.045426  0.139027  0.196587  0.000000   
accuracy_with_tol_1       mean  0.823390  0.931187  0.950684  0.000000   
                          std   0.051925  0.030649  0.015281  0.000000   
accuracy_with_tol_2       mean  0.827103  0.932113  0.954396  0.000000   
                          std   0.051449  0.031228  0.014944  0.000000   
accuracy_with_tol_3       mean  0.830841  0.933048  0.956265  0.000000   
                          std   0.051297  0.031756  0.013977  0.000000   
accuracy_with_tol_4       mean  0.839218  0.940498  0.983255  0.000000   
                          std   0.050801  0.026962  0.013748  0.000000   
average_levenshtein       mean  1.589339  0.568051  0.255858  0.685575   
                          std   0.564617  0.275778  0.105661  0.118518   
average_similarity        mean  0.856521  0.932678  0.951514  0.000000   
                          std   0.047629  0.030816  0.015097  0.000000   
average_levenshtein_match mean  1.777726  0.617256  0.269898  0.000000   
                          std   0.690156  0.314992  0.114956  0.000000   
average_similarity_match  mean  0.945896  0.998622  0.998921  0.000000   
                          std   0.019828  0.002528  0.003411  0.000000   
no_match_rate             mean 

In [37]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

prompt0_template = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_0.txt").read_text())

with FoldEvaluator("Qwen3.5-9B-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.


Fold 1 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Qwen3.5-9B-prompt0-0shot - Metrics for Country City Street House:


accuracy                   mean    0.895360
                           std     0.010901
precision                  mean    0.859612
                           std     0.019023
recall                     mean    0.818855
                           std     0.020621
f1                         mean    0.838674
                           std     0.018257
accuracy_with_tol_1        mean    0.901179
                           std     0.010211
accuracy_with_tol_2        mean    0.906525
                           std     0.010180
accuracy_with_tol_3        mean    0.916295
                           std     0.011246
accuracy_with_tol_4        mean    0.927466
                           std     0.010471
average_levenshtein        mean    0.718092
                           std     0.120217
average_similarity         mean    0.919957
                           std     0.010629
average_levenshtein_match  mean    0.762734
                           std     0.135021
average_similarity_match   mean 

Qwen3.5-9B-prompt0-0shot - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.952605    0.914417      0.893977   
                          std      0.017644    0.027352      0.033616   
precision                 mean     0.879931    0.817927      0.366535   
                          std      0.049646    0.047746      0.152542   
recall                    mean     0.889203    0.819884      0.392033   
                          std      0.043193    0.052617      0.158993   
f1                        mean     0.884422    0.818637      0.376215   
                          std      0.045377    0.048044      0.151125   
accuracy_with_tol_1       mean     0.955400    0.925606      0.895829   
                          std      0.017277    0.020906      0.035464   
accuracy_with_tol_2       mean     0.973053    0.926532      0.905140   
                          std      0.013399    0.022438      0.037289   
accuracy_with_tol_3       mean     0.985125    0.930253      0.906066   
                          std      0.010900    0.024390      0.038222   
accuracy_with_tol_4       mean     0.991641    0.941433      0.908844   
                          std      0.009211    0.022628      0.036821   
average_levenshtein       mean     0.162600    0.710038      0.867792   
                          std      0.082917    0.279172      0.370155   
average_similarity        mean     0.965336    0.947484      0.902583   
                          std      0.015034    0.020372      0.033486   
average_levenshtein_match mean     0.166238    0.737136      0.963919   
                          std      0.086696    0.302984      0.452624   
average_similarity_match  mean     0.981775    0.977451      0.988018   
                          std      0.009093    0.008635      0.006526   
no_match_rate             mean     0.016710    0.030659      0.086483   
                          std      0.014994    0.019025      0.033039   

Field/Entity                        City     State  District   Country  \
Metric                                                                   
accuracy                  mean  0.786068  0.933982  0.866978  0.928349   
                          std   0.026841  0.018727  0.024388  0.021075   
precision                 mean  0.863233  0.490490  0.309574  0.889508   
                          std   0.040392  0.083124  0.116266  0.068140   
recall                    mean  0.796758  0.905278  0.373518  0.792191   
                          std   0.028631  0.105136  0.153772  0.050585   
f1                        mean  0.828443  0.633103  0.332916  0.836703   
                          std   0.031370  0.083489  0.120851  0.047710   
accuracy_with_tol_1       mean  0.789780  0.936777  0.869773  0.933930   
                          std   0.024367  0.018909  0.021838  0.016757   
accuracy_with_tol_2       mean  0.790715  0.939573  0.871634  0.935800   
                          std   0.023534  0.019626  0.023052  0.018901   
accuracy_with_tol_3       mean  0.802821  0.943294  0.875363  0.946980   
                          std   0.025266  0.018174  0.022326  0.017501   
accuracy_with_tol_4       mean  0.813058  0.947015  0.889313  0.963733   
                          std   0.024328  0.019962  0.023295  0.022028   
average_levenshtein       mean  1.636951  0.479751  1.002838  0.362781   
                          std   0.211727  0.176496  0.245957  0.149186   
average_similarity        mean  0.828544  0.936191  0.883041  0.938466   
                          std   0.019573  0.018005  0.028377  0.018148   
average_levenshtein_match mean  1.874901  0.513925  1.122111  0.386760   
                          std   0.261043  0.194630  0.308702  0.165008   
average_similarity_match  mean  0.947629  0.997378  0.979866  0.993953   
                          std   0.012302  0.003196  0.013200  0.005593   
no_match_rate             mean 

In [38]:
prompt_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_optuna_best.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Llama-3-8B-best-from-optuna", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.LlamaAddressParsingModel(
            model_name="meta-llama/Meta-Llama-3-8B-Instruct", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 1 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 2 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 3 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 4 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 5 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 6 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 7 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 8 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Fold 9 already has predictions for Llama-3-8B-best-from-optuna, loading from file.
Llama-3-8B-best-from-optuna - Metrics for Country City Street House:


accuracy                   mean    0.942143
                           std     0.017998
precision                  mean    0.903240
                           std     0.026007
recall                     mean    0.930771
                           std     0.020947
f1                         mean    0.916688
                           std     0.021237
accuracy_with_tol_1        mean    0.946796
                           std     0.017733
accuracy_with_tol_2        mean    0.950283
                           std     0.018506
accuracy_with_tol_3        mean    0.955856
                           std     0.015373
accuracy_with_tol_4        mean    0.960272
                           std     0.016427
average_levenshtein        mean    0.387351
                           std     0.135505
average_similarity         mean    0.954569
                           std     0.016894
average_levenshtein_match  mean    0.402159
                           std     0.146126
average_similarity_match   mean 

Llama-3-8B-best-from-optuna - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.961890    0.959121      0.931231   
                          std      0.012670    0.014533      0.033880   
precision                 mean     0.905693    0.927331      0.622797   
                          std      0.029305    0.025845      0.180586   
recall                    mean     0.935619    0.925149      0.635714   
                          std      0.030252    0.037300      0.179133   
f1                        mean     0.920213    0.925944      0.623744   
                          std      0.026157    0.026950      0.171331   
accuracy_with_tol_1       mean     0.966554    0.960047      0.931231   
                          std      0.014573    0.013769      0.033880   
accuracy_with_tol_2       mean     0.976774    0.961907      0.932165   
                          std      0.012541    0.014044      0.032657   
accuracy_with_tol_3       mean     0.986042    0.967489      0.932165   
                          std      0.010090    0.015207      0.032657   
accuracy_with_tol_4       mean     0.990697    0.971210      0.933100   
                          std      0.008791    0.017121      0.031663   
average_levenshtein       mean     0.141295    0.365931      0.718242   
                          std      0.085177    0.192721      0.345001   
average_similarity        mean     0.971952    0.968189      0.933436   
                          std      0.011231    0.015522      0.032943   
average_levenshtein_match mean     0.144083    0.376592      0.777686   
                          std      0.087190    0.201543      0.402334   
average_similarity_match  mean     0.988533    0.992172      0.995358   
                          std      0.007757    0.006206      0.004743   
no_match_rate             mean     0.016727    0.024152      0.062245   
                          std      0.012965    0.015856      0.031768   

Field/Entity                        City   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.907961  0.939598                 0.923170   
                          std   0.041178  0.026195                 0.025902   
precision                 mean  0.918372  0.827773                 0.914902   
                          std   0.035819  0.059319                 0.028079   
recall                    mean  0.926758  0.941408                 0.928060   
                          std   0.039513  0.049739                 0.024554   
f1                        mean  0.922490  0.879220                 0.921403   
                          std   0.036947  0.039642                 0.025805   
accuracy_with_tol_1       mean  0.912608  0.947975                 0.929349   
                          std   0.037971  0.024676                 0.022352   
accuracy_with_tol_2       mean  0.913543  0.948910                 0.935565   
                          std   0.038680  0.024682                 0.021650   
accuracy_with_tol_3       mean  0.914477  0.955417                 0.947418   
                          std   0.036803  0.023758                 0.019197   
accuracy_with_tol_4       mean  0.916329  0.962850                 0.953028   
                          std   0.036090  0.023867                 0.021428   
average_levenshtein       mean  0.710549  0.331629                 0.505454   
                          std   0.293431  0.174679                 0.215098   
average_similarity        mean  0.930743  0.947391                 0.944840   
                          std   0.030674  0.025527                 0.020505   
average_levenshtein_match mean  0.746975  0.353444                 0.523908   
                          std   0.327988  0.200272                 0.227385   
average_similarity_match  mean  0.970295  0.995506         

In [39]:
supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Mistral-3-8B-Instruct", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.MistralAddressParsingModel(
            model_name="mistralai/Ministral-3-8B-Instruct-2512", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 1 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 2 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 3 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 4 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 5 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 6 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 7 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 8 already has predictions for Mistral-3-8B-Instruct, loading from file.
Fold 9 already has predictions for Mistral-3-8B-Instruct, loading from file.
Mistral-3-8B-Instruct - Metrics for Country City Street House:


accuracy                   mean    0.943527
                           std     0.015591
precision                  mean    0.915612
                           std     0.019720
recall                     mean    0.912416
                           std     0.018749
f1                         mean    0.913957
                           std     0.017795
accuracy_with_tol_1        mean    0.949569
                           std     0.014242
accuracy_with_tol_2        mean    0.953520
                           std     0.013546
accuracy_with_tol_3        mean    0.960027
                           std     0.011346
accuracy_with_tol_4        mean    0.963519
                           std     0.012414
average_levenshtein        mean    0.378117
                           std     0.104513
average_similarity         mean    0.957573
                           std     0.013446
average_levenshtein_match  mean    0.390460
                           std     0.112074
average_similarity_match   mean 

Mistral-3-8B-Instruct - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.965628    0.952605      0.917307   
                          std      0.013056    0.024538      0.031464   
precision                 mean     0.918342    0.916820      0.524103   
                          std      0.029689    0.044568      0.084447   
recall                    mean     0.930146    0.908253      0.686676   
                          std      0.023114    0.037380      0.131837   
f1                        mean     0.924078    0.912310      0.591507   
                          std      0.024147    0.038602      0.094105   
accuracy_with_tol_1       mean     0.967489    0.956317      0.918242   
                          std      0.013911    0.020079      0.033058   
accuracy_with_tol_2       mean     0.978626    0.957252      0.918242   
                          std      0.008767    0.020616      0.033058   
accuracy_with_tol_3       mean     0.988837    0.965620      0.918242   
                          std      0.008538    0.017499      0.033058   
accuracy_with_tol_4       mean     0.993501    0.969341      0.920093   
                          std      0.007630    0.019593      0.031421   
average_levenshtein       mean     0.129102    0.387383      0.807459   
                          std      0.062197    0.198310      0.313331   
average_similarity        mean     0.973737    0.966494      0.920645   
                          std      0.011294    0.018741      0.031254   
average_levenshtein_match mean     0.131104    0.400207      0.881157   
                          std      0.063381    0.209029      0.368895   
average_similarity_match  mean     0.987525    0.991380      0.992586   
                          std      0.007597    0.006707      0.004698   
no_match_rate             mean     0.013924    0.025069      0.072456   
                          std      0.012523    0.019538      0.031918   

Field/Entity                        City   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.894946  0.960929                 0.913674   
                          std   0.032911  0.023846                 0.020480   
precision                 mean  0.917986  0.906202                 0.910407   
                          std   0.027274  0.072713                 0.016875   
recall                    mean  0.906911  0.911710                 0.904439   
                          std   0.031040  0.047879                 0.020019   
f1                        mean  0.912340  0.907993                 0.907359   
                          std   0.027973  0.053497                 0.017013   
accuracy_with_tol_1       mean  0.905166  0.969306                 0.923878   
                          std   0.031963  0.019095                 0.018057   
accuracy_with_tol_2       mean  0.907035  0.971166                 0.930643   
                          std   0.032593  0.017747                 0.020825   
accuracy_with_tol_3       mean  0.907970  0.977683                 0.947754   
                          std   0.033589  0.015273                 0.022381   
accuracy_with_tol_4       mean  0.911682  0.979552                 0.953937   
                          std   0.033250  0.015001                 0.021351   
average_levenshtein       mean  0.796166  0.199818                 0.515817   
                          std   0.257181  0.134596                 0.183531   
average_similarity        mean  0.919858  0.970200                 0.939798   
                          std   0.027024  0.018175                 0.018249   
average_levenshtein_match mean  0.844825  0.206611                 0.536129   
                          std   0.292103  0.140513                 0.195850   
average_similarity_match  mean  0.969359  0.995172         

In [40]:
supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("DeepSeek-R1-Llama-8B", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.DeepSeekAddressParsingModel(
            model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 1 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 2 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 3 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 4 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 5 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 6 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 7 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 8 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
Fold 9 already has predictions for DeepSeek-R1-Llama-8B, loading from file.
DeepSeek-R1-Llama-8B - Metrics for Country City Street House:


accuracy                   mean      0.030577
                           std       0.066339
precision                  mean      0.500000
                           std       0.471405
recall                     mean      0.003864
                           std       0.004664
f1                         mean      0.007651
                           std       0.009194
accuracy_with_tol_1        mean      0.030577
                           std       0.066339
accuracy_with_tol_2        mean      0.030811
                           std       0.066242
accuracy_with_tol_3        mean      0.032667
                           std       0.068905
accuracy_with_tol_4        mean      0.041721
                           std       0.073960
average_levenshtein        mean      3.776904
                           std       0.218562
average_similarity         mean      0.030951
                           std       0.066205
average_levenshtein_match  mean     99.040496
                           std    

DeepSeek-R1-Llama-8B - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
precision                 mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
recall                    mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
f1                        mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_1       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_2       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_3       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_4       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
average_levenshtein       mean     0.942437    4.718744      0.982849   
                          std      0.195532    0.695158      0.277238   
average_similarity        mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
average_levenshtein_match mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
average_similarity_match  mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
no_match_rate             mean     1.000000    1.000000      1.000000   
                          std      0.000000    0.000000      0.000000   

Field/Entity                          City   Country  ApplicantCurrentAddress  \
Metric                                                                          
accuracy                  mean    0.049161  0.073148                 0.022573   
                          std     0.048194  0.231315                 0.056772   
precision                 mean    0.500000  0.100000                 0.000000   
                          std     0.471405  0.316228                 0.000000   
recall                    mean    0.007020  0.003333                 0.000000   
                          std     0.006943  0.010541                 0.000000   
f1                        mean    0.013823  0.006452                 0.000000   
                          std     0.013634  0.020402                 0.000000   
accuracy_with_tol_1       mean    0.049161  0.073148                 0.022573   
                          std     0.048194  0.231315                 0.056772   
accuracy_with_tol_2       mean    0.050095  0.073148                 0.022573   
                          std     0.047625  0.231315                 0.056772   
accuracy_with_tol_3       mean    0.053816  0.076852                 0.027230   
                          std     0.048732  0.243027                 0.065240   
accuracy_with_tol_4       mean    0.087253  0.079630                 0.036724   
                          std     0.071722  0.251811                 0.069347   
average_levenshtein       mean    7.828262  1.618172                 5.805845   
                          std     0.436561  0.341947                 0.209629   
average_similarity        mean    0.050655  0.073148                 0.022573   
                          std     0.047758  0.231315                 0.056772   
average_levenshtein_match mean  103.366667  0.192405                56.730702   
                          std   108.768634  0.608438               107.536648   
average_sim

In [41]:
supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Qwen3.5-9B-best-from-optuna", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Qwen3.5-9B-best-from-optuna - Metrics for Country City Street House:


accuracy                   mean    0.955136
                           std     0.011355
precision                  mean    0.927202
                           std     0.017988
recall                     mean    0.930435
                           std     0.014618
f1                         mean    0.928792
                           std     0.015614
accuracy_with_tol_1        mean    0.959320
                           std     0.010956
accuracy_with_tol_2        mean    0.960951
                           std     0.010912
accuracy_with_tol_3        mean    0.966295
                           std     0.008851
accuracy_with_tol_4        mean    0.970020
                           std     0.007619
average_levenshtein        mean    0.312188
                           std     0.077178
average_similarity         mean    0.966598
                           std     0.008885
average_levenshtein_match  mean    0.319164
                           std     0.081673
average_similarity_match   mean 

Qwen3.5-9B-best-from-optuna - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.971166    0.963768      0.914495   
                          std      0.009252    0.017154      0.032549   
precision                 mean     0.932043    0.936583      0.510210   
                          std      0.030627    0.036791      0.125769   
recall                    mean     0.934631    0.925679      0.644505   
                          std      0.022769    0.039026      0.213824   
f1                        mean     0.933210    0.930753      0.566442   
                          std      0.024486    0.033113      0.158919   
accuracy_with_tol_1       mean     0.974888    0.965620      0.916346   
                          std      0.008825    0.015781      0.032038   
accuracy_with_tol_2       mean     0.979543    0.966554      0.919124   
                          std      0.010546    0.016461      0.027959   
accuracy_with_tol_3       mean     0.990689    0.971184      0.919124   
                          std      0.008791    0.014823      0.027959   
accuracy_with_tol_4       mean     0.994418    0.974913      0.925640   
                          std      0.004804    0.014553      0.025016   
average_levenshtein       mean     0.106006    0.336345      0.754898   
                          std      0.054829    0.179742      0.236869   
average_similarity        mean     0.979391    0.973224      0.918064   
                          std      0.006972    0.013392      0.029312   
average_levenshtein_match mean     0.106972    0.343911      0.825190   
                          std      0.055175    0.185975      0.284180   
average_similarity_match  mean     0.988599    0.991636      0.994831   
                          std      0.006032    0.006599      0.004151   
no_match_rate             mean     0.009303    0.018579      0.077146   
                          std      0.006202    0.010684      0.029959   

Field/Entity                        City   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.913525  0.972084                 0.928175   
                          std   0.017896  0.015795                 0.029763   
precision                 mean  0.919659  0.932656                 0.924335   
                          std   0.016783  0.050232                 0.033936   
recall                    mean  0.925086  0.947195                 0.924097   
                          std   0.014302  0.041299                 0.031002   
f1                        mean  0.922332  0.939295                 0.924152   
                          std   0.014483  0.039112                 0.031471   
accuracy_with_tol_1       mean  0.919107  0.977665                 0.933486   
                          std   0.017366  0.012555                 0.027907   
accuracy_with_tol_2       mean  0.919107  0.978600                 0.938014   
                          std   0.017366  0.012431                 0.028337   
accuracy_with_tol_3       mean  0.920976  0.982330                 0.949307   
                          std   0.017977  0.011117                 0.022909   
accuracy_with_tol_4       mean  0.923771  0.986976                 0.957500   
                          std   0.018758  0.008977                 0.021980   
average_levenshtein       mean  0.671452  0.134952                 0.487113   
                          std   0.160774  0.078487                 0.225860   
average_similarity        mean  0.936615  0.977160                 0.947370   
                          std   0.016406  0.012572                 0.022613   
average_levenshtein_match mean  0.695782  0.138670                 0.504163   
                          std   0.178611  0.081064                 0.237382   
average_similarity_match  mean  0.967196  0.998529         

In [67]:
def style_results(df : pd.DataFrame) -> pd.io.formats.style.Styler:
    styled_df = df.apply(
        lambda row: pd.Series({
            col: f"{row[(col, 'mean')]:.2f}±{row[(col, 'std')]:.2f}"
            for col in row.index.get_level_values(0)
        }, name=row.name),
        axis=1
    ).style.apply(
        lambda s: [
            'font-weight: bold' 
            if df.at[k, (s.name, 'mean')].round(2) == df[(s.name, 'mean')].max().round(2)
            else '' 
            for k in s.index
        ],
        axis=0
    )
    return styled_df

In [68]:
all_experiments = pd.DataFrame({k : v[["f1", "precision", "recall", "accuracy"]] for k, v in required_results.items()}).T

all_experiments_styled = style_results(all_experiments)

display(all_experiments_styled)

,f1,precision,recall,accuracy
libpostal,0.51±0.04,0.64±0.05,0.42±0.04,0.69±0.03
deepparse,0.29±0.03,0.32±0.03,0.27±0.04,0.57±0.03
xlm-roberta-large,0.68±0.03,0.71±0.04,0.66±0.03,0.81±0.02
SpaCy,0.83±0.04,0.85±0.03,0.81±0.04,0.89±0.03
Qwen3.5-9B-prompt0-0shot,0.84±0.02,0.86±0.02,0.82±0.02,0.90±0.01
Llama-3-8B-best-from-optuna,0.92±0.02,0.90±0.03,0.93±0.02,0.94±0.02
Mistral-3-8B-Instruct,0.91±0.02,0.92±0.02,0.91±0.02,0.94±0.02
DeepSeek-R1-Llama-8B,0.01±0.01,0.50±0.47,0.00±0.00,0.03±0.07
Qwen3.5-9B-best-from-optuna,0.93±0.02,0.93±0.02,0.93±0.01,0.96±0.01


In [69]:
# Subset of experiments selected for inclusion in the paper.
selected_experiments = all_experiments

# Sanity check that the selected experiments include all the best results
assert all(all_experiments[c].argmax() == selected_experiments[c].argmax() for c in all_experiments.columns), "Selected experiments must have the same best metrics as all experiments for the required entities"

selected_experiments_styled = style_results(selected_experiments)

display(selected_experiments_styled)

,f1,precision,recall,accuracy
libpostal,0.51±0.04,0.64±0.05,0.42±0.04,0.69±0.03
deepparse,0.29±0.03,0.32±0.03,0.27±0.04,0.57±0.03
xlm-roberta-large,0.68±0.03,0.71±0.04,0.66±0.03,0.81±0.02
SpaCy,0.83±0.04,0.85±0.03,0.81±0.04,0.89±0.03
Qwen3.5-9B-prompt0-0shot,0.84±0.02,0.86±0.02,0.82±0.02,0.90±0.01
Llama-3-8B-best-from-optuna,0.92±0.02,0.90±0.03,0.93±0.02,0.94±0.02
Mistral-3-8B-Instruct,0.91±0.02,0.92±0.02,0.91±0.02,0.94±0.02
DeepSeek-R1-Llama-8B,0.01±0.01,0.50±0.47,0.00±0.00,0.03±0.07
Qwen3.5-9B-best-from-optuna,0.93±0.02,0.93±0.02,0.93±0.01,0.96±0.01


In [71]:
print(
    selected_experiments_styled.to_latex(hrules=True, convert_css=True)
      .replace("& f1 &", "& F$_1$ &")
      .replace("±", "$\\pm$")
)

\begin{tabular}{lllll}
\toprule
 & F$_1$ & precision & recall & accuracy \\
\midrule
libpostal & 0.51$\pm$0.04 & 0.64$\pm$0.05 & 0.42$\pm$0.04 & 0.69$\pm$0.03 \\
deepparse & 0.29$\pm$0.03 & 0.32$\pm$0.03 & 0.27$\pm$0.04 & 0.57$\pm$0.03 \\
xlm-roberta-large & 0.68$\pm$0.03 & 0.71$\pm$0.04 & 0.66$\pm$0.03 & 0.81$\pm$0.02 \\
SpaCy & 0.83$\pm$0.04 & 0.85$\pm$0.03 & 0.81$\pm$0.04 & 0.89$\pm$0.03 \\
Qwen3.5-9B-prompt0-0shot & 0.84$\pm$0.02 & 0.86$\pm$0.02 & 0.82$\pm$0.02 & 0.90$\pm$0.01 \\
Llama-3-8B-best-from-optuna & 0.92$\pm$0.02 & 0.90$\pm$0.03 & \bfseries 0.93$\pm$0.02 & 0.94$\pm$0.02 \\
Mistral-3-8B-Instruct & 0.91$\pm$0.02 & 0.92$\pm$0.02 & 0.91$\pm$0.02 & 0.94$\pm$0.02 \\
DeepSeek-R1-Llama-8B & 0.01$\pm$0.01 & 0.50$\pm$0.47 & 0.00$\pm$0.00 & 0.03$\pm$0.07 \\
Qwen3.5-9B-best-from-optuna & \bfseries 0.93$\pm$0.02 & \bfseries 0.93$\pm$0.02 & \bfseries 0.93$\pm$0.01 & \bfseries 0.96$\pm$0.01 \\
\bottomrule
\end{tabular}

